## Merging with FRED Data

1. Load datasets

In [ ]:
import pandas as pd

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

sold = pd.read_csv("..\\processed\\CRMLSSold_Residential_Cleaned.csv", low_memory=False)
listings = pd.read_csv("..\\processed\\CRMLSListing_Residential_Cleaned.csv", low_memory=False)

2. Resample weekly rates to monthly averages

In [2]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')

mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)

3. Create a matching year_month key on the MLS datasets

In [3]:
# Sold dataset — key off CloseDate

sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')

# Listings dataset — key off ListingContractDate
listings['year_month'] = pd.to_datetime(listings['ListingContractDate']
                                        ).dt.to_period('M')

4. Merge data

In [4]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

5. Validate merge

In [9]:
# Check for any unmatched rows (rate should not be null)
print(f"Sold unmatched rows: {sold_with_rates['rate_30yr_fixed'].isnull().sum()}")
print(f"Listings unmatched rows: {listings_with_rates['rate_30yr_fixed'].isnull().sum()}")

# Preview
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
    ].head()
)

print(f"Sold median close price: ${sold_with_rates['ClosePrice'].median():,.0f}")
print(f"Sold mean close price: ${sold_with_rates['ClosePrice'].mean():,.0f}")

Sold unmatched rows: 0
Listings unmatched rows: 0
    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-26    2024-01    240000.0           6.6425
1  2024-01-05    2024-01    815000.0           6.6425
2  2024-01-05    2024-01    810000.0           6.6425
3  2024-01-02    2024-01   2100000.0           6.6425
4  2024-01-22    2024-01   1950000.0           6.6425
Sold median close price: $825,000
Sold mean close price: $1,189,588


6. Save enriched datasets

In [ ]:
sold_with_rates.to_csv("..\\processed\\CRMLSSold_Residential_With_Rates.csv", index=False)
listings_with_rates.to_csv("..\\processed\\CRMLSListing_Residential_With_Rates.csv", index=False)